## Hospital Patient Tracking System 

### Scenario
You're building a system to track patients, their appointments, and doctors in a small clinic. You need to create the database structure, populate it with realistic data, then analyze patient load and doctor schedules.

In [1]:
# 1. Create Database & Tables
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, inspect

# Create/connect to hospital database
conn = sqlite3.connect("hospital.db")
engine = create_engine("sqlite:///hospital.db")

In [11]:
# Create tables
conn.execute("""
CREATE TABLE IF NOT EXISTS doctors (
    doctor_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    specialty TEXT NOT NULL,
    hire_date DATE
)
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS patients (
    patient_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    age INTEGER,
    phone TEXT,
    registration_date DATE
)
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS appointments (
    appointment_id INTEGER PRIMARY KEY,
    patient_id INTEGER,
    doctor_id INTEGER,
    appointment_date DATE,
    status TEXT,
    FOREIGN KEY (patient_id) REFERENCES patients (patient_id),
    FOREIGN KEY (doctor_id) REFERENCES doctors (doctor_id)
)
""")

In [13]:
# 2. Insert Sample Data
# Doctors
doctors_data = [
    (1, "Dr. Sarah Johnson", "Cardiology", "2023-01-15"),
    (2, "Dr. Michael Chen", "Neurology", "2022-06-10"),
    (3, "Dr. Emily Rodriguez", "Pediatrics", "2024-03-01"),
    (4, "Dr. John Chen", "Cardiology", "2024-02-01")
]

conn.executemany("INSERT OR REPLACE INTO doctors VALUES (?,?,?,?)", doctors_data)

# Patients
patients_data = [
    (1, "John Smith", 45, "416-555-0101", "2025-01-10"),
    (2, "Maria Garcia", 32, "416-555-0202", "2025-02-15"),
    (3, "Ahmed Khan", 28, "416-555-0303", "2025-03-01"),
    (4, "Sophie Lee", 8, "416-555-0404", "2025-04-20"),
    (5, "Jane Doe", 21, "212-238-4107", "2025-04-20")
]

conn.executemany("INSERT OR REPLACE INTO patients VALUES (?,?,?, ?, ?)", patients_data)

# Appointments
appointments_data = [
    (1, 1, 1, "2026-05-15", "Scheduled"),
    (2, 2, 2, "2026-05-16", "Scheduled"),
    (3, 3, 1, "2026-05-17", "Completed"),
    (4, 4, 3, "2026-05-18", "Scheduled"),
    (5, 1, 2, "2026-05-20", "Scheduled"),
    (6, 5, 4, "2026-05-22", "Scheduled")
]

conn.executemany("INSERT OR REPLACE INTO appointments VALUES (?,?,?,?,?)", appointments_data)
conn.commit()
conn.close()

In [14]:
# 3. Explore the Data
# Check tables exist
inspector = inspect(engine)
print("Tables:", inspector.get_table_names())
# Output: ['doctors', 'patients', 'appointments']

# Doctors
doctors_df = pd.read_sql("SELECT * FROM doctors", engine)
print("Doctors:")
print(doctors_df)

Tables: ['appointments', 'doctors', 'patients']
Doctors:
   doctor_id                 name   specialty   hire_date
0          1    Dr. Sarah Johnson  Cardiology  2023-01-15
1          2     Dr. Michael Chen   Neurology  2022-06-10
2          3  Dr. Emily Rodriguez  Pediatrics  2024-03-01
3          4        Dr. John Chen  Cardiology  2024-02-01


In [15]:
# Patients by age group
patients_df = pd.read_sql("SELECT * FROM patients", engine)
patients_df['age_group'] = pd.cut(patients_df['age'], bins=[0, 18, 65, 100], labels=['Child', 'Adult', 'Senior'])
print("Patients:")
print(patients_df)

Patients:
   patient_id          name  age         phone registration_date age_group
0           1    John Smith   45  416-555-0101        2025-01-10     Adult
1           2  Maria Garcia   32  416-555-0202        2025-02-15     Adult
2           3    Ahmed Khan   28  416-555-0303        2025-03-01     Adult
3           4    Sophie Lee    8  416-555-0404        2025-04-20     Child
4           5      Jane Doe   21  212-238-4107        2025-04-20     Adult


# 4. Real-World Analysis Queries

In [16]:
# Doctor workload
workload = pd.read_sql("""
    SELECT d.name, d.specialty, COUNT(a.appointment_id) as appointments
    FROM doctors d
    LEFT JOIN appointments a ON d.doctor_id = a.doctor_id
    GROUP BY d.doctor_id
""", engine)
print("Doctor Workload:")
print(workload)

Doctor Workload:
                  name   specialty  appointments
0    Dr. Sarah Johnson  Cardiology             2
1     Dr. Michael Chen   Neurology             2
2  Dr. Emily Rodriguez  Pediatrics             1
3        Dr. John Chen  Cardiology             1


In [17]:
# Appointments this week
this_week = pd.read_sql("""
    SELECT p.name, d.name as doctor, a.appointment_date, a.status
    FROM appointments a
    JOIN patients p ON a.patient_id = p.patient_id
    JOIN doctors d ON a.doctor_id = d.doctor_id
    WHERE a.appointment_date >= '2026-05-15'
""", engine)
print("This Week's Appointments:")
print(this_week)

This Week's Appointments:
           name               doctor appointment_date     status
0    John Smith    Dr. Sarah Johnson       2026-05-15  Scheduled
1  Maria Garcia     Dr. Michael Chen       2026-05-16  Scheduled
2    Ahmed Khan    Dr. Sarah Johnson       2026-05-17  Completed
3    Sophie Lee  Dr. Emily Rodriguez       2026-05-18  Scheduled
4    John Smith     Dr. Michael Chen       2026-05-20  Scheduled
5      Jane Doe        Dr. John Chen       2026-05-22  Scheduled


## Task 1: CSV Loading Basics
Download the NBA Elo dataset (~126K rows of basketball stats) from a public URL and load it into a DataFrame. Use pd.read_csv() with parameters to skip the first row if needed, then display shape, data types (df.dtypes), and first 5 rows (df.head()).

In [36]:

import pandas as pd
import requests
from io import StringIO

url = "https://raw.githubusercontent.com/fivethirtyeight/data/master/nba-elo/nbaallelo.csv"

response = requests.get(url, verify=False)

data = StringIO(response.text)

nba_elo = pd.read_csv(data)

print(nba_elo.shape)
print(nba_elo.dtypes)
print(nba_elo.head())

(126314, 23)
gameorder          int64
game_id              str
lg_id                str
_iscopy            int64
year_id            int64
date_game            str
seasongame         int64
is_playoffs        int64
team_id              str
fran_id              str
pts                int64
elo_i            float64
elo_n            float64
win_equiv        float64
opp_id               str
opp_fran             str
opp_pts            int64
opp_elo_i        float64
opp_elo_n        float64
game_location        str
game_result          str
forecast         float64
notes                str
dtype: object
   gameorder       game_id lg_id  _iscopy  year_id  date_game  seasongame  \
0          1  194611010TRH   NBA        0     1947  11/1/1946           1   
1          1  194611010TRH   NBA        1     1947  11/1/1946           1   
2          2  194611020CHS   NBA        0     1947  11/2/1946           1   
3          2  194611020CHS   NBA        1     1947  11/2/1946           2   
4          3 

# Task 2: JSON Dataset Import
Fetch a sample JSON dataset like GitHub user data or a public API response (e.g., JSONPlaceholder todos). Load with pd.read_json()

In [40]:
import pandas as pd
import requests

url_json = "https://gist.githubusercontent.com/saltukalakus/124bba04327d8e5eab605d4fb66c53b8/raw/sample_users_with_id.json"

response = requests.get(url_json, verify=False)

df_json = pd.DataFrame(response.json())

print(df_json.head())
print(df_json.dtypes)
print(df_json.shape)


/Users/rukku/Documents/Sheridan/Python/PythonProject/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'gist.githubusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
